In [21]:
!pip install -q google-genai pandas

In [22]:
import pandas as pd
from google import genai

In [25]:
client = genai.Client(
    api_key="your api key"
)

MODEL_NAME = "gemini-flash-latest"

In [26]:
transactions = [

{
"Transaction_ID":"TXN1001",
"Customer_Name":"Rahul Sharma",
"Amount":2500,
"Payment_Method":"Credit Card",
"Bank":"HDFC",
"Status":"Success",
"Failure_Reason":"-"
},

{
"Transaction_ID":"TXN1002",
"Customer_Name":"Priya Verma",
"Amount":1800,
"Payment_Method":"UPI",
"Bank":"SBI",
"Status":"Success",
"Failure_Reason":"-"
},

{
"Transaction_ID":"TXN1003",
"Customer_Name":"Aman Gupta",
"Amount":3200,
"Payment_Method":"Debit Card",
"Bank":"ICICI",
"Status":"Failed",
"Failure_Reason":"Insufficient Balance"
},

{
"Transaction_ID":"TXN1004",
"Customer_Name":"Neha Reddy",
"Amount":950,
"Payment_Method":"UPI",
"Bank":"Axis",
"Status":"Success",
"Failure_Reason":"-"
},

{
"Transaction_ID":"TXN1005",
"Customer_Name":"Rohan Mehta",
"Amount":5400,
"Payment_Method":"Credit Card",
"Bank":"HDFC",
"Status":"Failed",
"Failure_Reason":"Payment Gateway Timeout"
},

{
"Transaction_ID":"TXN1006",
"Customer_Name":"Ananya Singh",
"Amount":760,
"Payment_Method":"Net Banking",
"Bank":"SBI",
"Status":"Success",
"Failure_Reason":"-"
},

{
"Transaction_ID":"TXN1007",
"Customer_Name":"Karan Patel",
"Amount":4300,
"Payment_Method":"Debit Card",
"Bank":"ICICI",
"Status":"Failed",
"Failure_Reason":"Bank Server Unavailable"
},

{
"Transaction_ID":"TXN1008",
"Customer_Name":"Sneha Iyer",
"Amount":1200,
"Payment_Method":"UPI",
"Bank":"Kotak",
"Status":"Success",
"Failure_Reason":"-"
},

{
"Transaction_ID":"TXN1009",
"Customer_Name":"Arjun Rao",
"Amount":8900,
"Payment_Method":"Credit Card",
"Bank":"Axis",
"Status":"Failed",
"Failure_Reason":"Suspicious Transaction Detected"
},

{
"Transaction_ID":"TXN1010",
"Customer_Name":"Meera Joshi",
"Amount":670,
"Payment_Method":"UPI",
"Bank":"HDFC",
"Status":"Success",
"Failure_Reason":"-"
}
]

df = pd.DataFrame(transactions)

df

,Transaction_ID,Customer_Name,Amount,Payment_Method,Bank,Status,Failure_Reason
0,TXN1001,Rahul Sharma,2500,Credit Card,HDFC,Success,-
1,TXN1002,Priya Verma,1800,UPI,SBI,Success,-
2,TXN1003,Aman Gupta,3200,Debit Card,ICICI,Failed,Insufficient Balance
3,TXN1004,Neha Reddy,950,UPI,Axis,Success,-
4,TXN1005,Rohan Mehta,5400,Credit Card,HDFC,Failed,Payment Gateway Timeout
5,TXN1006,Ananya Singh,760,Net Banking,SBI,Success,-
6,TXN1007,Karan Patel,4300,Debit Card,ICICI,Failed,Bank Server Unavailable
7,TXN1008,Sneha Iyer,1200,UPI,Kotak,Success,-
8,TXN1009,Arjun Rao,8900,Credit Card,Axis,Failed,Suspicious Transaction Detected
9,TXN1010,Meera Joshi,670,UPI,HDFC,Success,-


In [27]:
failed = df[df["Status"]=="Failed"]

failed

,Transaction_ID,Customer_Name,Amount,Payment_Method,Bank,Status,Failure_Reason
2,TXN1003,Aman Gupta,3200,Debit Card,ICICI,Failed,Insufficient Balance
4,TXN1005,Rohan Mehta,5400,Credit Card,HDFC,Failed,Payment Gateway Timeout
6,TXN1007,Karan Patel,4300,Debit Card,ICICI,Failed,Bank Server Unavailable
8,TXN1009,Arjun Rao,8900,Credit Card,Axis,Failed,Suspicious Transaction Detected


In [28]:
for i,row in failed.iterrows():

    prompt=f"""
You are a Senior FinTech Risk Analyst.

Analyze the following failed payment transaction.

Transaction ID : {row['Transaction_ID']}
Customer : {row['Customer_Name']}
Amount : {row['Amount']}
Payment Method : {row['Payment_Method']}
Bank : {row['Bank']}
Failure Reason : {row['Failure_Reason']}

Return ONLY in this format.

Root Cause:
Risk Level:
Explanation:
Suggested Fixes:
1.
2.
3.
"""

    response=client.models.generate_content(

        model=MODEL_NAME,

        contents=prompt

    )

    print("="*80)
    print("Transaction :",row["Transaction_ID"])
    print("="*80)

    print(response.text)

    print()

Transaction : TXN1003
Root Cause: The customer's linked ICICI Bank account had insufficient cleared funds to cover the transaction amount of 3200.
Risk Level: Low (Standard customer-side financial decline; no indicators of fraud, system downtime, or security breach).
Explanation: The transaction was processed through the payment gateway to the issuing bank (ICICI), which returned a standard ISO 8583 Response Code 51 (Insufficient Funds). This is a user-end soft decline, meaning the payment infrastructure functioned correctly, but the transaction could not be authorized due to the account's ledger balance.
Suggested Fixes:
1. Implement real-time in-app dunning that prompts the user to switch to an alternative payment method (such as UPI, Credit Card, or NetBanking) to complete the transaction immediately.
2. Display a clear, jargon-free error message to the customer indicating that the transaction failed due to insufficient funds, advising them to top up their account.
3. If this is a r

In [29]:
summary=[]

for _,row in failed.iterrows():

    summary.append({

        "Transaction ID":row["Transaction_ID"],

        "Customer":row["Customer_Name"],

        "Amount":row["Amount"],

        "Failure Reason":row["Failure_Reason"]

    })

summary_df=pd.DataFrame(summary)

summary_df

,Transaction ID,Customer,Amount,Failure Reason
0,TXN1003,Aman Gupta,3200,Insufficient Balance
1,TXN1005,Rohan Mehta,5400,Payment Gateway Timeout
2,TXN1007,Karan Patel,4300,Bank Server Unavailable
3,TXN1009,Arjun Rao,8900,Suspicious Transaction Detected


In [30]:
summary_df.to_csv(

    "payment_failure_report.csv",

    index=False

)

print("Report Generated Successfully.")

Report Generated Successfully.


In [31]:
from google.colab import files

files.download(

    "payment_failure_report.csv"

)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
print("="*70)
print("AI Payment Failure Analyzer")
print("="*70)

print("Total Transactions :",len(df))
print("Failed Payments :",len(failed))

print("\nDetected Failed Transactions")

for _,row in summary_df.iterrows():

    print("-",row["Transaction ID"],":",row["Failure Reason"])

print("\nAnalysis Completed Successfully.")

AI Payment Failure Analyzer
Total Transactions : 10
Failed Payments : 4

Detected Failed Transactions
- TXN1003 : Insufficient Balance
- TXN1005 : Payment Gateway Timeout
- TXN1007 : Bank Server Unavailable
- TXN1009 : Suspicious Transaction Detected

Analysis Completed Successfully.
